In [ ]:
import numpy as np
import pandas as pd

from utils import data_utils as du
from utils import stats_utils as su

cities = du.load_major_us_cities()
sf_raw = cities["sf_Existing_Buildings_Energy_Performance_Ordinance_Report_20260114"]
sea_raw = cities[
    "Seattle_Benchmarking_Performance_Ranges_by_Building_Type_2015-Present_20260125"
]
bos_raw = cities["boston_energy_raw"]

energy_data = du.load_data()
concurrent_df = du.concurrent_buildings(
    input_df=energy_data,
    start_year=2016,
    end_year=2023,
)

2026-02-21 18:49:27,252 [INFO] Loaded Seattle_Benchmarking_Performance_Ranges_by_Building_Type_2015-Present_20260125.csv → (34699, 46)
2026-02-21 18:49:27,374 [INFO] Loaded sf_Existing_Buildings_Energy_Performance_Ordinance_Report_20260114.csv → (28243, 34)
2026-02-21 18:49:59,891 [INFO] Loaded Boston_data folder → (35496, 96)


In [73]:
sf_chi = du.sf_to_chicago(sf_raw)
sea_chi = du.seattle_to_chicago(sea_raw)
sf_chi = sf_chi.assign(City="San Francisco")
sea_chi = sea_chi.assign(City="Seattle")
bos = du.harmonize_boston_columns(bos_raw)
bos_chi = du.boston_to_chicago(bos).assign(City="Boston")
bos_chi["Data Year"] = pd.to_numeric(bos_chi["Data Year"]) - 1

## DiD

In [74]:
did_df = su.build_multi_city_did_df(
    chicago_df=concurrent_df,
    other_city_dfs={
        "San Francisco": sf_chi,
        "Seattle": sea_chi,
        "Boston": bos_chi,
    },
    start_year=2016,
    end_year=2023,
    post_start_year=2020,
    outcome_col="Site EUI (kBtu/sq ft)",
)

/project/src/utils/stats_utils.py:423: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  did_df = pd.concat(frames, ignore_index=True)
/project/src/utils/stats_utils.py:423: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  did_df = pd.concat(frames, ignore_index=True)
/project/.venv/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


### three cities as baseline on ALL building types

$$
y_{it} = \beta_0 
+ \beta_1 \text{Chicago}_i 
+ \beta_2 \text{Post2019}_t 
+ \beta_3 (\text{Chicago}_i \times \text{Post2019}_t) 
+ \gamma_t 
+ \delta_c 
+ \theta_{k(i)} 
+ \varepsilon_{it}
$$


In [82]:
did_df = did_df.copy()

did_df["Site EUI (kBtu/sq ft)"] = pd.to_numeric(
    did_df["Site EUI (kBtu/sq ft)"], errors="coerce"
).astype("float64")

for c in ["Post", "LowRating", "Interaction"]:
    did_df[c] = pd.to_numeric(did_df[c], errors="coerce").fillna(0).astype("int64")

did_df["Data Year"] = pd.to_numeric(did_df["Data Year"], errors="coerce")
did_df = did_df.dropna(subset=["Data Year"])
did_df["Data Year"] = did_df["Data Year"].astype("int64")

did_df["Primary Property Type"] = (
    did_df["Primary Property Type"].astype(str).str.strip().str.lower()
)
if "City" in did_df.columns:
    did_df["City"] = did_df["City"].astype("object")

did_df = did_df.dropna(
    subset=["Site EUI (kBtu/sq ft)", "ln_FloorArea", "Primary Property Type"]
)

In [81]:
model_eui = su.run_did_regression(
    did_df,
    "Site EUI (kBtu/sq ft)",
    include_data_year=True,
)

su.summarize_did_results(model_eui, focus_terms=["Post", "LowRating", "Interaction"])

/project/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 101, but rank is 1
  warnings.warn('covariance of constraints does not have full '
2026-02-21 19:36:08,172 [INFO] Showing 9 selected coefficients (policy + energy types).


,coef,std_err,p_value,Significance
C(Q('Primary Property Type'))[T.data center],4.517641e+02,5.120530e+01,0.0000,***
C(Q('Primary Property Type'))[T.fitness center/health club/gym],2.015890e+01,1.257270e+01,0.1089,
C(Q('Primary Property Type'))[T.hospital (general medical & surgical)],2.201403e+02,6.154040e+01,0.0003,***
C(Q('Primary Property Type'))[T.laboratory],2.125362e+02,2.849340e+01,0.0000,***
C(Q('Primary Property Type'))[T.other - specialty hospital],1.501889e+02,4.045880e+01,0.0002,***
C(Q('Primary Property Type'))[T.other/specialty hospital],9.340660e+01,2.766280e+01,0.0007,***
Post,-5.102558e+12,5.896885e+12,0.3869,
LowRating,-2.475690e+01,1.707990e+01,0.1472,
Interaction,5.613510e+01,2.897350e+01,0.0527,*


#### log scale

In [83]:
did_df = did_df.copy()

did_df["Site EUI (kBtu/sq ft)"] = pd.to_numeric(
    did_df["Site EUI (kBtu/sq ft)"], errors="coerce"
).astype("float64")

did_df = did_df[did_df["Site EUI (kBtu/sq ft)"] > 0]

did_df["ln_Site_EUI"] = np.log(did_df["Site EUI (kBtu/sq ft)"])

In [84]:
model_eui_log = su.run_did_regression(
    did_df,
    "ln_Site_EUI",
    include_data_year=True,
)

su.summarize_did_results(
    model_eui_log, focus_terms=["Post", "LowRating", "Interaction"]
)

/project/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 101, but rank is 1
  warnings.warn('covariance of constraints does not have full '
2026-02-21 19:36:35,124 [INFO] Showing 9 selected coefficients (policy + energy types).


,coef,std_err,p_value,Significance
C(Q('Primary Property Type'))[T.data center],1.906800e+00,1.299000e-01,0.0000,***
C(Q('Primary Property Type'))[T.fitness center/health club/gym],1.546000e-01,9.310000e-02,0.0969,*
C(Q('Primary Property Type'))[T.hospital (general medical & surgical)],1.174000e+00,7.680000e-02,0.0000,***
C(Q('Primary Property Type'))[T.laboratory],1.329000e+00,7.590000e-02,0.0000,***
C(Q('Primary Property Type'))[T.other - specialty hospital],6.101000e-01,2.296000e-01,0.0079,***
C(Q('Primary Property Type'))[T.other/specialty hospital],6.153000e-01,1.821000e-01,0.0007,***
Post,1.109834e+10,3.011234e+10,0.7125,
LowRating,3.984000e-01,8.600000e-03,0.0000,***
Interaction,1.580000e-02,9.400000e-03,0.0915,*


#### filter by concurrent

In [20]:
sf_concurrent = du.concurrent_buildings(
    input_df=sf_chi,
    start_year=2016,
    end_year=2024,
    id_col="Address",
    status_col="__IGNORE_STATUS__",
)

sea_concurrent = du.concurrent_buildings(
    input_df=sea_chi,
    start_year=2016,
    end_year=2024,
    id_col="Address",
    status_col="__IGNORE_STATUS__",
)

bos_concurrent = du.concurrent_buildings(
    input_df=bos_chi,
    start_year=2016,
    end_year=2024,
    id_col="Address",
    status_col="__IGNORE_STATUS__",
)

In [85]:
did_df_lvl = did_df.copy()

# Outcome in levels
did_df_lvl["Site EUI (kBtu/sq ft)"] = pd.to_numeric(
    did_df_lvl["Site EUI (kBtu/sq ft)"], errors="coerce"
).astype("float64")

# Core DiD terms as plain int64 (NOT pandas Int64)
for c in ["Post", "LowRating", "Interaction"]:
    did_df_lvl[c] = (
        pd.to_numeric(did_df_lvl[c], errors="coerce").fillna(0).astype("int64")
    )

# Controls
did_df_lvl["ln_FloorArea"] = pd.to_numeric(
    did_df_lvl["ln_FloorArea"], errors="coerce"
).astype("float64")

# Year as plain int64 (important for patsy)
did_df_lvl["Data Year"] = pd.to_numeric(did_df_lvl["Data Year"], errors="coerce")
did_df_lvl = did_df_lvl.dropna(subset=["Data Year"])
did_df_lvl["Data Year"] = did_df_lvl["Data Year"].astype("int64")

# Categoricals
did_df_lvl["Primary Property Type"] = (
    did_df_lvl["Primary Property Type"].astype(str).str.strip().str.lower()
)
if "City" in did_df_lvl.columns:
    did_df_lvl["City"] = did_df_lvl["City"].astype("object")

# Drop missing essentials
did_df_lvl = did_df_lvl.dropna(
    subset=[
        "Site EUI (kBtu/sq ft)",
        "ln_FloorArea",
        "Primary Property Type",
        "Data Year",
    ]
)

In [86]:
model_eui_lvl = su.run_did_regression(
    did_df_lvl,
    "Site EUI (kBtu/sq ft)",
    include_data_year=True,
)

su.summarize_did_results(
    model_eui_lvl,
    focus_terms=["Post", "LowRating", "Interaction"],
)

/project/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 101, but rank is 1
  warnings.warn('covariance of constraints does not have full '
2026-02-21 19:43:27,757 [INFO] Showing 9 selected coefficients (policy + energy types).


,coef,std_err,p_value,Significance
C(Q('Primary Property Type'))[T.data center],4.517641e+02,5.120530e+01,0.0000,***
C(Q('Primary Property Type'))[T.fitness center/health club/gym],2.015890e+01,1.257270e+01,0.1089,
C(Q('Primary Property Type'))[T.hospital (general medical & surgical)],2.201403e+02,6.154040e+01,0.0003,***
C(Q('Primary Property Type'))[T.laboratory],2.125362e+02,2.849340e+01,0.0000,***
C(Q('Primary Property Type'))[T.other - specialty hospital],1.501889e+02,4.045880e+01,0.0002,***
C(Q('Primary Property Type'))[T.other/specialty hospital],9.340660e+01,2.766280e+01,0.0007,***
Post,-5.102558e+12,5.896885e+12,0.3869,
LowRating,-2.475690e+01,1.707990e+01,0.1472,
Interaction,5.613510e+01,2.897350e+01,0.0527,*


### three cities as baseline on Multifamily Housing/Office

In [ ]:
def make_concurrent_panel(
    panel: pd.DataFrame, id_col: str, year_col: str = "Data Year"
) -> pd.DataFrame:
    """Return a balanced (concurrent) panel where each building appears in all years."""
    panel = panel.copy()
    panel[year_col] = pd.to_numeric(panel[year_col], errors="coerce")
    panel = panel.dropna(subset=[id_col, year_col])
    panel[year_col] = panel[year_col].astype("int64")

    years = sorted(panel[year_col].unique())
    n_years = len(years)

    n_by_building = panel.groupby(id_col)[year_col].nunique()
    keep_ids = n_by_building[n_by_building == n_years].index
    return panel[panel[id_col].isin(keep_ids)].copy()

In [122]:
mf_df = su.filter_property_type(did_df, "multifamily")
mf_df = su.prep_for_did_levels(mf_df)

In [143]:
id_col = "ID" if "ID" in did_df.columns else "Address"

office_df = su.filter_property_type(did_df, "office")
office_df = su.prep_for_did_levels(office_df)

office_conc = su.make_concurrent_panel(office_df, id_col=id_col)

model_office = su.run_did_regression(
    office_conc,
    "Site EUI (kBtu/sq ft)",
    include_data_year=True,
)

su.summarize_did_results(
    model_office,
    focus_terms=["Post", "LowRating", "Interaction"],
)

/project/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 13, but rank is 12
  warnings.warn('covariance of constraints does not have full '
2026-02-21 20:27:04,511 [INFO] Showing 3 selected coefficients (policy + energy types).


,coef,std_err,p_value,Significance
Post,-7.0994,1.5078,0.0000,***
LowRating,27.3601,1.6419,0.0000,***
Interaction,-3.5779,1.9201,0.0624,*


In [144]:
id_col = "ID" if "ID" in did_df.columns else "Address"

mf_df = su.filter_property_type(did_df, "multifamily")
mf_df = su.prep_for_did_levels(mf_df)

mf_conc = su.make_concurrent_panel(mf_df, id_col=id_col)

model_mf = su.run_did_regression(
    mf_conc,
    "Site EUI (kBtu/sq ft)",
    include_data_year=True,
)

su.summarize_did_results(
    model_mf,
    focus_terms=["Post", "LowRating", "Interaction"],
)

/project/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 17, but rank is 1
  warnings.warn('covariance of constraints does not have full '
2026-02-21 20:27:23,334 [INFO] Showing 3 selected coefficients (policy + energy types).


,coef,std_err,p_value,Significance
Post,-4.150468e+12,2.466238e+12,0.0924,*
LowRating,4.110170e+01,4.405400e+00,0.0000,***
Interaction,-5.507600e+00,7.486000e-01,0.0000,***
